## 0. Imports and Configurations

In [58]:
import os
from dotenv import load_dotenv
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor

load_dotenv()

USE_S3 = os.getenv("USE_S3", "false").lower() == "true"

## Load Cleaned Dataset

In [59]:
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, '..', 'data')

# Dynamic route to the data file
if USE_S3:
    df = pd.read_csv(f"s3://{os.getenv('S3_BUCKET')}/cleaned_data.csv")
else:
    df = pd.read_csv(os.path.join(DATA_DIR, 'cleaned_data.csv'))

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)

BASE_DIR: /home/admin/Documents/nextbuy/notebooks
DATA_DIR: /home/admin/Documents/nextbuy/notebooks/../data


## Define target

In [60]:
basket_size = df.groupby('order_id').size().reset_index(name='basket_size')
print(basket_size.head())

   order_id  basket_size
0         2            9
1         4           13
2         6            3
3         8            1
4        21            5


## Orders dataset

In [61]:
orders = df.groupby('order_id').first().reset_index()

orders = orders.merge(basket_size, on='order_id')

## Feature preparation

In [62]:
features = [
    # variables temporelles
    "order_dow",
    "order_hour_of_day",
    "days_since_prior_order",
    # historique client
    "order_number",
    "is_first_order",
    "basket_size",  # ← variable cible
    # comportement d'achat

    # "reordered_count",
    # "reorder_ratio",
    # "n_aisles",
    # "n_departments",
]

target = "basket_size"

df_model = orders[features + [target]]
print(df_model[target])

         basket_size  basket_size
0                  9            9
1                 13           13
2                  3            3
3                  1            1
4                  5            5
...              ...          ...
1270556            8            8
1270557            3            3
1270558           13           13
1270559            5            5
1270560            8            8

[1270561 rows x 2 columns]


## Train test split

In [63]:
X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Taille de l'ensemble d'entraînement :", X_train.shape)
print("Taille de l'ensemble de test :", X_test.shape)

Taille de l'ensemble d'entraînement : (1016448, 7)
Taille de l'ensemble de test : (254113, 7)


## Baseline

In [64]:
model = LinearRegression()
model.fit(X_train, y_train)

pred = model.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(rmse, mae, r2)

7.059040434150326e-15 5.3165248970982846e-15 1.0


## Random Forest

In [65]:
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)

rmse = np.sqrt(mean_squared_error(y_test, pred))
mae = mean_absolute_error(y_test, pred)
r2 = r2_score(y_test, pred)

print(rmse, mae, r2)

0.013962339360707758 5.8705971617781255e-05 0.9999965628878947
